### Ensemble Classifiers (Random Forest)

In [1]:
import pandas as pd
import time
import warnings
import tracemalloc
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import f1_score
from scipy.stats import loguniform

In [3]:
def random_forest(data):
    y = data['collision']
    x = data.drop('collision', axis=1)
    
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=81)

    rf = RandomForestClassifier(random_state=81)

    tracemalloc.start()
    start_time = time.time()

    rf.fit(x_train, y_train)

    training_time = time.time() - start_time
    _, peak_memory = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    y_pred = rf.predict(x_test)
    
    f1_metric = f1_score(y_test, y_pred)
    peak_memory_mb = peak_memory / (1024 * 1024)

    return f1_metric, training_time, peak_memory_mb

#### Mетрики без подбора гиперпараметров

In [4]:
n_samples = [100, 500, 1000, 3000]
m_features = [5, 8, 11]

results = []

for n in n_samples:
    for m in m_features:
        data = pd.read_csv(f"f1_data/f1_data_{n}_s_{m}_f.csv")
        f1, real_time, mem_usage = random_forest(data)
        results.append({
            'samples (n)' : n,
            'features (m)' : m,
            'f1-score' : f1,
            'time (sec)' : real_time,
            'memory (MB)': mem_usage
        })

results_df = pd.DataFrame(results)

f1_avg = results_df['f1-score'].mean()
real_time_avg = results_df['time (sec)'].mean()
mem_usage_avg = results_df['memory (MB)'].mean()

print(f'Средняя f1-мера = {round(f1_avg, 3)}')
print(f'Среднее время = {round(real_time_avg, 5)} сек')
print(f'Среднее потребление памяти = {round(mem_usage_avg, 3)} MB')

results_df

Средняя f1-мера = 0.993
Среднее время = 0.2513 сек
Среднее потребление памяти = 0.241 MB


,samples (n),features (m),f1-score,time (sec),memory (MB)
0,100,5,1.000000,0.240007,0.196853
1,100,8,1.000000,0.220450,0.103301
2,100,11,0.965517,0.218217,0.107575
3,500,5,0.993377,0.225115,0.142697
4,500,8,0.983333,0.245424,0.146764
5,500,11,1.000000,0.239323,0.163119
6,1000,5,1.000000,0.233265,0.195004
7,1000,8,1.000000,0.234191,0.199390
8,1000,11,0.984520,0.255818,0.241541
9,3000,5,1.000000,0.258736,0.402700


Метод ближайших соседей показывает лучший `f1-score`, но требует больше памяти.

#### Mетрики с подбором гиперпараметров

In [7]:
def random_forest_params(data):
    y = data['collision']
    x = data.drop('collision', axis=1)
    
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=81)

    param_dist = {
        'n_estimators': [50, 100, 200],          # количество деревьев
        'max_depth': [10, 20, 30],         # максимальная глубина
        'min_samples_split': [2, 5, 10],         # порог разделения узла
        'criterion': ['gini', 'entropy']         # способ измерения качества разделения
    }

    rf = RandomForestClassifier(random_state=81)

    random_search = RandomizedSearchCV(
        rf, param_dist, n_iter=10, cv=5, scoring='f1', random_state=81
    )

    tracemalloc.start()
    start_time = time.time()

    random_search.fit(x_train, y_train)

    training_time = time.time() - start_time
    _, peak_memory = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    best_model = random_search.best_estimator_
    y_pred = best_model.predict(x_test)
    
    f1_metric = f1_score(y_test, y_pred)
    best_params = random_search.best_params_
    peak_memory_mb = peak_memory / (1024 * 1024)

    return f1_metric, training_time, peak_memory_mb, best_params

In [8]:
results = []

for n in n_samples:
    for m in m_features:
        data = pd.read_csv(f"f1_data/f1_data_{n}_s_{m}_f.csv")
        f1, real_time, mem_usage, best_p = random_forest_params(data)
        results.append({
            'samples (n)' : n,
            'features (m)' : m,
            'f1-score' : f1,
            'time (sec)' : real_time,
            'memory (MB)': mem_usage,
            'n_estimators': best_p['n_estimators'],
            'max_depth': best_p['max_depth'],
            'min_samples_split': best_p['min_samples_split'],
            'criterion': best_p['criterion']
        })

results_df = pd.DataFrame(results)

f1_avg = results_df['f1-score'].mean()
real_time_avg = results_df['time (sec)'].mean()
mem_usage_avg = results_df['memory (MB)'].mean()

print(f'Средняя f1-мера = {round(f1_avg, 3)}')
print(f'Среднее время = {round(real_time_avg, 5)} сек')
print(f'Среднее потребление памяти = {round(mem_usage_avg, 3)} MB')

results_df

Средняя f1-мера = 0.992
Среднее время = 19.2353 сек
Среднее потребление памяти = 0.502 MB


,samples (n),features (m),f1-score,time (sec),memory (MB),n_estimators,max_depth,min_samples_split,criterion
0,100,5,1.000000,17.345703,0.251750,200,10,10,entropy
1,100,8,1.000000,17.640350,0.240149,200,10,10,entropy
2,100,11,0.965517,18.097081,0.256767,50,30,5,gini
3,500,5,0.993377,18.132664,0.313464,200,10,10,entropy
4,500,8,0.975207,17.928514,0.326569,50,30,2,entropy
5,500,11,0.987013,19.021032,0.368957,200,10,10,entropy
6,1000,5,1.000000,18.333590,0.411648,200,10,10,entropy
7,1000,8,0.996283,18.719723,0.460757,200,10,10,entropy
8,1000,11,0.987654,19.750703,0.544148,50,20,5,entropy
9,3000,5,1.000000,19.790944,0.796331,200,10,10,entropy


После подбора гиперпараметра `f1-score` незначительно снизился, затраты памяти возросли. 

Количество деревьев `n_estimators` - задает число независимых деревьев решений в лесу. Чем больше деревьев в лесу, тем более стабильным и точным становится итоговое предсказание, так как результат определяется путем «голосования» большинства.

Минимум объектов для деления `min_samples_split` - минимальное количество строк датасета, которое должно находиться в узле, чтобы алгоритм мог создать из него новое разветвление.

Критерий информативности `criterion` - математическая функция, которая измеряет качество разделения данных в каждом узле дерева (используются `gini` — индекс Джини или `entrop`y — информационная энтропия).